# 01 — PySpark lineage with Spline

Reads the 100-row NYC Yellow Taxi sample, transforms it via the
DataFrame API, and writes both an Iceberg table and a Parquet sink.
Each persistent write produces a Spline lineage event that you can
inspect at <http://localhost:9090>.

**Note**: Spline's stock Spark agent detects most write commands but
the Iceberg `CreateTableAsSelect` flow wraps the plan in an
Iceberg-specific node that the agent does not currently expose.
So we also write a Parquet sink so the lineage is guaranteed to
appear in Spline UI. The Iceberg table is still the artifact used
for downstream queries.


In [ ]:
from pyspark.sql import functions as F
from _shared.spark_session import get_spark, SAMPLE_CSV, PARQUET_SINK

spark = get_spark()
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


In [ ]:
taxi_pdf = spark.read.csv(SAMPLE_CSV, header=True, inferSchema=True)
print('row count:', taxi_pdf.count())
taxi_pdf.printSchema()
taxi_pdf.show(5, truncate=False)


## Build the analytics layers

Two derived frames:

* `trip_durations` — adds a duration column and keeps numeric metrics.
* `zone_revenue` — aggregates pickup-zone revenue.


In [ ]:
trip_durations = (
    taxi_pdf
    .withColumn(
        'trip_minutes',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60.0,
    )
    .select(
        'PULocationID', 'DOLocationID', 'payment_type',
        'trip_distance', 'trip_minutes', 'fare_amount', 'tip_amount', 'total_amount',
    )
)
trip_durations.show(5, truncate=False)

zone_revenue = (
    taxi_pdf
    .groupBy('PULocationID')
    .agg(
        F.count('*').alias('trips'),
        F.round(F.sum('fare_amount'), 2).alias('revenue'),
        F.round(F.avg('tip_amount'), 2).alias('avg_tip'),
    )
    .orderBy(F.desc('revenue'))
)
zone_revenue.show(5, truncate=False)


## Persist lineage

Both writes are persistent actions, so Spline will emit a lineage
event for each. The Iceberg table is the analytics-grade artifact,
the Parquet sink guarantees Spline captures the column-level
transformations.


In [ ]:
(trip_durations.write
 .format('iceberg')
 .mode('overwrite')
 .saveAsTable('local.taxi.trip_durations'))

# Spline-supported sink — always visible in Spline UI.
(zone_revenue.write
 .mode('overwrite')
 .parquet(f'{PARQUET_SINK}/zone_revenue'))

print('wrote trip_durations (iceberg) and zone_revenue (parquet)')


## Confirm what Spline sees

The producer is on `spline-rest:8080`. After these writes, open
<http://localhost:9090> and look for the latest execution events.
The `zone_revenue` write should show a full column-level graph
from `yellow_trip_sample.csv` to the Parquet sink.


In [ ]:
import urllib.request, json
events = json.loads(urllib.request.urlopen('http://spline-rest:8080/consumer/execution-events').read())
events = events.get('items', events)
print('captured events:', len(events))
for e in events[:5]:
    print(' -', e.get('name'), '|', e.get('id'))
